# C · Demo and real photos

- a Gradio app with a public link: pick a test person or upload your own, add any shirt
- also tests photos from outside the dataset (needs detectron2, step 7)
- your own Colab account · **Runtime → Change runtime type → T4 GPU**, then Run all
- output: `MyDrive/viton/results_C/`

## 1. Settings

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

ROLE = 'C'
GUIDANCE = 2.5
H, W = 512, 384
SEED = 42
DEV = 'cuda'
SAVE_TO_DRIVE = True  # keeps outputs if colab disconnects
STEPS = 30

## 2. Drive

In [ ]:
import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = f'/content/drive/MyDrive/viton/results_{ROLE}'
else:
    OUT = f'/content/results_{ROLE}'
os.makedirs(OUT, exist_ok=True)
print('outputs ->', OUT)

## 3. Install and get CatVTON code

In [ ]:
!pip -q install diffusers accelerate transformers peft torchmetrics torch-fidelity lpips gdown gradio
!rm -rf /content/CatVTON
!GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 https://huggingface.co/spaces/zhengchong/CatVTON /content/CatVTON

import sys
sys.path.insert(0, '/content/CatVTON')
os.chdir('/content/CatVTON')
print(sorted(os.listdir('.')))

## Get VITON-HD

- official link from the VITON-HD repo (google drive)
- if gdown fails with a quota error: open the link in a browser, add it to your drive, put the zip at `MyDrive/viton/zalando-hd-resized.zip`, run this cell again

In [ ]:
import glob, shutil
ZIP = '/content/viton.zip'
DRIVE_ZIP = '/content/drive/MyDrive/viton/zalando-hd-resized.zip'

if not os.path.exists(ZIP):
    if os.path.exists(DRIVE_ZIP):
        shutil.copy(DRIVE_ZIP, ZIP)
    else:
        !gdown 1tLx8LRp-sxDp0EcYmYoV_vXdSc-jJ79w -O {ZIP}

size_gb = os.path.getsize(ZIP) / 1e9 if os.path.exists(ZIP) else 0
print(f'zip: {size_gb:.2f} GB')
assert size_gb > 1, 'download failed, see the note above this cell'

!unzip -q -o {ZIP} -x '*train/*' -d /content/vitonhd

TEST = os.path.dirname(glob.glob('/content/vitonhd/**/test/image', recursive=True)[0])
PAIRS = glob.glob('/content/vitonhd/**/test_pairs.txt', recursive=True)[0]
print('test folder:', TEST)
print('subfolders:', sorted(os.listdir(TEST)))
!head -3 {PAIRS}

## 5. Masks

In [ ]:
import numpy as np, cv2, random, time, json
from PIL import Image
import matplotlib.pyplot as plt

def mask_from_agnostic(name):
    img = np.asarray(Image.open(f'{TEST}/image/{name}').convert('RGB')).astype(int)
    agn = np.asarray(Image.open(f'{TEST}/agnostic-v3.2/{name}').convert('RGB')).astype(int)
    m = (np.abs(img - agn).sum(2) > 40).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    m = cv2.dilate(m, np.ones((9, 9), np.uint8))
    return Image.fromarray(m * 255)

def mask_from_parse(name):
    p = np.asarray(Image.open(f'{TEST}/image-parse-v3/{name.replace(".jpg", ".png")}'))
    m = np.isin(p, [5, 6, 7, 14, 15]).astype(np.uint8)  # upper clothes, dress, coat, arms
    m = cv2.dilate(m, np.ones((25, 25), np.uint8))
    return Image.fromarray(m * 255)

USE_AGNOSTIC = os.path.isdir(f'{TEST}/agnostic-v3.2')
get_mask = mask_from_agnostic if USE_AGNOSTIC else mask_from_parse
print('mask source:', 'agnostic-v3.2' if USE_AGNOSTIC else 'image-parse-v3')

names = sorted(os.listdir(f'{TEST}/image'))
with open(PAIRS) as f:
    pair_lines = [l.split() for l in f if l.strip()]
print(len(names), 'test people,', len(pair_lines), 'unpaired pairs')

def show_row(ax, imgs, titles):
    for a, im, t in zip(ax, imgs, titles):
        a.imshow(im, cmap='gray'); a.set_title(t, fontsize=9); a.axis('off')

fig, ax = plt.subplots(2, 4, figsize=(12, 8))
for i, n in enumerate(names[:4]):
    ax[0, i].imshow(Image.open(f'{TEST}/image/{n}')); ax[0, i].set_title(n); ax[0, i].axis('off')
    ax[1, i].imshow(get_mask(n), cmap='gray'); ax[1, i].axis('off')
plt.tight_layout(); plt.show()

## 6. Load CatVTON

In [ ]:
import torch, types, functools, tqdm as _tqdm
from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor
import model.pipeline as mp
from model.pipeline import CatVTONPipeline
from utils import resize_and_crop, resize_and_padding

mp.tqdm = types.SimpleNamespace(tqdm=functools.partial(_tqdm.tqdm, disable=True))  # quiet per-step bars

REPO = snapshot_download('zhengchong/CatVTON', allow_patterns=['vitonhd-16k-512/*', 'DensePose/*', 'SCHP/*'])
pipe = CatVTONPipeline(
    base_ckpt='booksforcharlie/stable-diffusion-inpainting',
    attn_ckpt=REPO,
    attn_ckpt_version='vitonhd',
    weight_dtype=torch.float16,
    device='cuda',
    skip_safety_check=True,
    use_tf32=False,
)
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)

def tryon(person, cloth, mask, steps=30, seed=SEED):
    person = resize_and_crop(person.convert('RGB'), (W, H))
    mask = resize_and_crop(mask.convert('L'), (W, H))
    cloth = resize_and_padding(cloth.convert('RGB'), (W, H))
    mask = mask_processor.blur(mask, blur_factor=9)
    g = torch.Generator(device='cuda').manual_seed(int(seed))
    out = pipe(image=person, condition_image=cloth, mask=mask, num_inference_steps=int(steps),
               guidance_scale=GUIDANCE, height=H, width=W, generator=g)[0]
    return out, person, cloth, mask

print('gpu memory used: %.1f GB' % (torch.cuda.memory_allocated() / 1e9))

## 7. Masks for any photo (detectron2)

- builds in about 5–10 min
- if it fails, the demo still works with test-set people

In [ ]:
import torch
torch.jit.script = lambda f: f  # same patch catvton uses before importing detectron2
!pip -q install fvcore iopath pycocotools omegaconf hydra-core yacs termcolor tabulate cloudpickle av
!pip -q install --no-build-isolation --no-deps 'git+https://github.com/facebookresearch/detectron2.git'

try:
    from model.cloth_masker import AutoMasker
    automasker = AutoMasker(densepose_ckpt=os.path.join(REPO, 'DensePose'),
                            schp_ckpt=os.path.join(REPO, 'SCHP'), device='cuda')
    print('automasker ready')
except Exception as e:
    automasker = None
    print('automasker not available, demo will use test-set people only:', e)

## 8. Demo

- open the `gradio.live` link it prints
- take a screenshot and a short screen recording as backup

In [ ]:
import gradio as gr
demo_people = random.Random(0).sample(names, 30)

def run(choice, upload, shirt, steps, seed):
    if shirt is None:
        raise gr.Error('upload a shirt photo')
    if upload is not None:
        if automasker is None:
            raise gr.Error('own photos need step 7; clear the photo and pick a test person')
        person = resize_and_crop(upload.convert('RGB'), (W, H))
        mask = automasker(person, 'upper')['mask']
    else:
        person = Image.open(f'{TEST}/image/{choice}')
        mask = get_mask(choice)
    out, person, _, mask = tryon(person, shirt, mask, steps, seed)
    shown = np.asarray(person).copy()
    shown[np.asarray(mask.convert('L')) > 127] = 128
    return Image.fromarray(shown), out

with gr.Blocks(title='Virtual Try-On') as demo:
    gr.Markdown('## Virtual Try-On · CatVTON on VITON-HD')
    with gr.Row():
        with gr.Column():
            choice = gr.Dropdown(demo_people, value=demo_people[0], label='test person')
            upload = gr.Image(type='pil', label='or your own person photo (optional)')
            shirt = gr.Image(type='pil', label='shirt photo')
            steps = gr.Slider(10, 50, value=STEPS, step=5, label='steps')
            seed = gr.Number(value=SEED, label='seed')
            go = gr.Button('Try on', variant='primary')
        with gr.Column():
            masked = gr.Image(label='area to repaint')
            result = gr.Image(label='result')
    gr.Examples([[demo_people[0], None, f'{TEST}/cloth/{c}', STEPS, SEED] for _, c in pair_lines[:4]],
                [choice, upload, shirt, steps, seed])
    go.click(run, [choice, upload, shirt, steps, seed], [masked, result])

demo.launch(share=True, debug=False)

## 9. Real photos

- upload person photos and shirt photos with the file picker
- every person is tried with every shirt, results saved to Drive

In [ ]:
if automasker is None:
    print('step 7 failed, so own photos are skipped')
else:
    from google.colab import files as colab_files
    os.makedirs(f'{OUT}/real', exist_ok=True)
    print('pick PERSON photos'); people_up = colab_files.upload()
    print('pick SHIRT photos'); shirts_up = colab_files.upload()
    for pn, pb in people_up.items():
        open(f'{OUT}/real/person_{pn}', 'wb').write(pb)
    for sn, sb in shirts_up.items():
        open(f'{OUT}/real/shirt_{sn}', 'wb').write(sb)

    rows = [(pn, sn) for pn in people_up for sn in shirts_up]
    fig, ax = plt.subplots(len(rows), 4, figsize=(11, 3.6 * len(rows)), squeeze=False)
    for r, (pn, sn) in enumerate(rows):
        person = resize_and_crop(Image.open(f'{OUT}/real/person_{pn}').convert('RGB'), (W, H))
        shirt = Image.open(f'{OUT}/real/shirt_{sn}').convert('RGB')
        mask = automasker(person, 'upper')['mask']
        out, person, shirt_p, mask = tryon(person, shirt, mask, steps=STEPS)
        out.save(f'{OUT}/real/result_{os.path.splitext(pn)[0]}__{os.path.splitext(sn)[0]}.png')
        show_row(ax[r], [person, shirt_p, mask, out], ['person', 'shirt', 'mask', 'try-on'])
    plt.tight_layout(); plt.savefig(f'{OUT}/real_photos.png', dpi=100); plt.show()